In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from utils.spark_session import get_spark

In [ ]:
HDFS_ROOT   = "hdfs://localhost:9000/rossmann"
OUTPUT_PATH = f"{HDFS_ROOT}/clean/df_clean.parquet"

In [ ]:
def load_data(spark):
    """Đọc sale.csv và store.csv từ HDFS, trả về 2 pandas DataFrame."""
    df_sales_spark  = spark.read.csv(f"{HDFS_ROOT}/sale.csv",
                                     header=True, inferSchema=True)
    df_stores_spark = spark.read.csv(f"{HDFS_ROOT}/store.csv",
                                     header=True, inferSchema=True)

    print(f"Sales  - {df_sales_spark.count():,} rows x {len(df_sales_spark.columns)} cols")
    print(f"Stores - {df_stores_spark.count():,} rows x {len(df_stores_spark.columns)} cols")

    df_sales  = df_sales_spark.toPandas()
    df_stores = df_stores_spark.toPandas()
    return df_sales, df_stores


In [ ]:
def clean_and_join(df_sales: pd.DataFrame,
                   df_stores: pd.DataFrame) -> pd.DataFrame:
    """Làm sạch missing values và join 2 bộ dữ liệu."""
    print("\n── Bộ dữ liệu Sales ──")
    print(df_sales.shape)
    print(df_sales.isnull().sum())

    print("\n── Bộ dữ liệu Stores ──")
    print(df_stores.shape)
    print(df_stores.isnull().sum())

    # Nhận thấy rằng chỉ có 3 mẫu trong biến CompetitionDistance là bị missing,
    # ta xử lý nó bằng cách fill giá trị trung bình của biến CompetitionDistance
    df_stores["CompetitionDistance"] = df_stores[
        "CompetitionDistance"
    ].fillna(df_stores["CompetitionDistance"].mean())

    # Những mẫu nào không có giá trị trong biến Promo2 thì ta sẽ fill bằng 0
    df_stores["Promo2"] = df_stores["Promo2"].fillna(0)

    # Join theo Store
    df_sales["Date"] = pd.to_datetime(df_sales["Date"])
    df = df_sales.merge(right=df_stores, on="Store", how="left")
    df["Date"] = pd.to_datetime(df["Date"])

    print(f"\n✅ Joined shape: {df.shape}")
    print(df.head())
    return df